In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

gemini_api_key = os.getenv("GEMINI_API_KEY")  # ensure this is set

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",     # <-- required
    google_api_key=gemini_api_key # optional if the lib reads env automatically
)

resp = model.invoke([HumanMessage(content="Hi! I am Aditya. How are you?")])
print(resp.content)

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Hi Aditya! I'm doing well, thank you for asking!

It's nice to meet you! How can I help you today?


In [3]:
resp2 = model.invoke([HumanMessage(content="What is my name?")])
print(resp2.content)

I don't know your name. As an AI, I don't have access to personal information about you.

If you'd like me to call you something, please feel free to tell me!


In [4]:
resp3 = model.invoke([
    HumanMessage(content="Hi! I am Aditya. How are you?"),
    AIMessage(content=resp.content),
    HumanMessage(content="What is my name?")
])

print(resp3.content)

Your name is Aditya! You told me that just a moment ago. 😊


In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system","You are a helpful assistant. Answer all the question to the best of you ability."
    ),
    MessagesPlaceholder(variable_name="messages"),
])

chain = prompt | model

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory

chat_message_history = ChatMessageHistory()

chat_message_history.add_user_message(
    "Transalate this sentence form English to French: I love programming."
)

response = chain.invoke(
    {"messages": chat_message_history.messages}
)

response

AIMessage(content="**J'aime la programmation.**", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--1af2624b-0cfc-4043-be68-66ee3889a02e-0', usage_metadata={'input_tokens': 31, 'output_tokens': 197, 'total_tokens': 228, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 189}})

In [7]:
chat_message_history.add_ai_message(response)
chat_message_history.add_user_message("What did I ask you?")

response = chain.invoke(
    {"messages": chat_message_history.messages}
)

print(response)

content='You asked me to translate the English sentence "I love programming" into French.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--cb51f4cf-a8f0-4f90-af44-2bc61eb607a5-0' usage_metadata={'input_tokens': 47, 'output_tokens': 51, 'total_tokens': 98, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 35}}


In [8]:
chat_message_history.messages

[HumanMessage(content='Transalate this sentence form English to French: I love programming.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="**J'aime la programmation.**", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--1af2624b-0cfc-4043-be68-66ee3889a02e-0', usage_metadata={'input_tokens': 31, 'output_tokens': 197, 'total_tokens': 228, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 189}}),
 HumanMessage(content='What did I ask you?', additional_kwargs={}, response_metadata={})]

In [9]:
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [10]:
from langchain_core.runnables.history import RunnableWithMessageHistory

model_with_message_history = RunnableWithMessageHistory(model, get_session_history)
# this is how we create a wrapper around our model to enable message history management

In [11]:
content = input("Enter your message: ")

config = {"configurable": {"session_id": "1"}}

model_with_message_history.invoke(
    [HumanMessage(content=content)],
    config=config
)

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--f8cb7deb-634e-474c-a17c-18135a4392cf-0', usage_metadata={'input_tokens': 2, 'output_tokens': 34, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 25}})

In [12]:
model_with_message_history.get_session_history("1").messages

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--f8cb7deb-634e-474c-a17c-18135a4392cf-0', usage_metadata={'input_tokens': 2, 'output_tokens': 34, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 25}})]

In [13]:
# chatbot using gradio
import gradio as gr


def preditct(message, history):
    config = {"configurable": {"session_id":"1122"}}
    model_with_message_history = RunnableWithMessageHistory(model, get_session_history)
    
    response1 = model_with_message_history.invoke(
        [HumanMessage(content=message)],
        config=config,
    ).content
    
    print(model_with_message_history.get_session_history("1122").messages)
    
    return response1



gr.ChatInterface(preditct).launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[HumanMessage(content='sdfa', additional_kwargs={}, response_metadata={}), AIMessage(content="It looks like you might have typed something by mistake, or perhaps you're just testing the system!\n\nIf you have a question or need help with something, please let me know. I'm ready to assist!", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--7f8a486a-dc8f-497c-a922-fcb2520db7dd-0', usage_metadata={'input_tokens': 3, 'output_tokens': 152, 'total_tokens': 155, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 107}})]
Keyboard interruption in main thread... closing server.


In [15]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system","You are a helpful assistant. Answer all the question to the best of you ability in {language}.",
    ),
    MessagesPlaceholder(variable_name="my_chat_history"),
    ("human","{input}")
])

chain = prompt | model
chain

ChatPromptTemplate(input_variables=['input', 'language', 'my_chat_history'], input_types={'my_chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], 

In [18]:
model_with_message_history = RunnableWithMessageHistory(chain, get_session_history,history_messages_key="my_chat_history", input_messages_key="input")

config = {"configurable": {"session_id": "222"}}

content = input("Enter your message: ")

result = model_with_message_history.invoke(
    {
        "input": content,
        "language": "Spanish",
    }, config=config
)

result

AIMessage(content='¡Claro! Madrid es una ciudad fascinante con una rica historia, cultura vibrante y una energía inigualable. Aquí te cuento un poco más sobre ella:\n\n1.  **Capital y Ubicación:** Madrid es la capital y la ciudad más grande de España. Se encuentra en el centro de la Península Ibérica, lo que la convierte en un importante nudo de comunicaciones para todo el país.\n\n2.  **Historia y Orígenes:** Aunque tiene orígenes que se remontan a la época romana y visigoda, su importancia real comenzó en el siglo IX con la construcción de una fortaleza musulmana. Sin embargo, su verdadero auge se dio en 1561, cuando el rey Felipe II la designó como capital del Imperio Español, desbancando a Toledo. Desde entonces, ha crecido y evolucionado hasta convertirse en la metrópolis que es hoy.\n\n3.  **Cultura y Arte:** Madrid es un epicentro cultural de primer orden. Alberga algunos de los museos más importantes del mundo:\n    *   **Museo del Prado:** Con una de las colecciones de arte eu

In [19]:
model_with_message_history.get_session_history("222").messages

[HumanMessage(content='What is the capital of spain?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='La capital de España es **Madrid**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--a707bb80-3d6c-48a3-a8ce-a42e0833c198-0', usage_metadata={'input_tokens': 27, 'output_tokens': 37, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 29}}),
 HumanMessage(content='tell me about it more', additional_kwargs={}, response_metadata={}),
 AIMessage(content='¡Claro! Madrid es una ciudad fascinante con una rica historia, cultura vibrante y una energía inigualable. Aquí te cuento un poco más sobre ella:\n\n1.  **Capital y Ubicación:** Madrid es la capital y la ciudad más grande de España. Se encuentra en el centro de la Península Ibérica, lo que la 

In [20]:
# to maintain combination of user id and session id for history management
from langchain_core.runnables import ConfigurableFieldSpec

store = {}

def get_session_history(user_id:str, conversation_id: str) -> ChatMessageHistory:
    if(user_id, conversation_id) not in store:
        store[(user_id, conversation_id)] = ChatMessageHistory()
    return store[(user_id, conversation_id)]

In [ ]:
model_with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key = "input",
    history_messages_key="my_chat_history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",annotation=str,name="User ID", is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",annotation=str,name="Conversation ID", is_shared=True,
        )
    ]
    )


config = {"configurable": {"user_id":"123","conversation_id":"1"}}

content = input("Enter your message: ")

result = model_with_message_history.invoke(
    {
        "input": content,
        "language": "Marathi",
    }, config=config
)

result

AIMessage(content='ਤੁਹਾਡਾ ਨਾਮ ਆਦਿਤਿਆ ਹੈ।', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--4068468e-2006-43cb-8e7a-281a09fd6944-0', usage_metadata={'input_tokens': 46, 'output_tokens': 13, 'total_tokens': 59, 'input_token_details': {'cache_read': 0}})

In [26]:
model_with_message_history.get_session_history("123","1").messages

[HumanMessage(content='My name is Aditya', additional_kwargs={}, response_metadata={}),
 AIMessage(content='नमस्कार, आदित्य!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--256eaf4d-c3e3-415d-9a1c-7ce7bf7847ab-0', usage_metadata={'input_tokens': 24, 'output_tokens': 330, 'total_tokens': 354, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 326}}),
 HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}),
 AIMessage(content='तुमचं नाव आदित्य आहे.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--57151bf3-df70-4c9e-a795-37e9809061c6-0', usage_metadata={'input_tokens': 